# Fixed synthetic U-Net workflow

Train a narrow U-Net on fixed geometric-shape samples, select on validation, then evaluate the saved model on the separate synthetic test split. This is not a real-world segmentation benchmark. No downloads are needed.

In [ ]:
import os
from pathlib import Path
import uuid
project = Path.cwd()
if not (project/'configs').is_dir():
    project = project.parent
assert (project/'configs').is_dir(), 'Launch from the project or notebooks directory'
os.chdir(project)
run = project/'artifacts'/('notebook-'+uuid.uuid4().hex[:12])

In [ ]:
from unet_yolo.segmentation import train,predict
result = train(project/'configs/unet-smoke.json',run)
print('Selected epoch:',result['best_epoch'])
print('Validation:',result['validation'])
print('Synthetic test:',result['test'])

In [ ]:
from IPython.display import display
from PIL import Image
display(Image.open(run/'training.png'))
display(Image.open(run/'predictions.png'))

## Reloaded-checkpoint inference at an odd rectangular resolution

The new image is still synthetic. The binary mask and probability array retain its original dimensions.

In [ ]:
import numpy as np
from unet_yolo.data import Shapes
image,_ = Shapes(1,(65,67),seed=123,split='test')[0]
image_path = run/'odd-shape.png'
Image.fromarray((image.permute(1,2,0).numpy()*255).astype(np.uint8)).save(image_path)
prediction = predict(run/'best.pt',image_path,run/'odd-prediction')
assert np.load(run/'odd-prediction/probabilities.npy').shape == (65,67)
display(Image.open(run/'odd-prediction/mask.png'))
print(prediction)

Metrics use per-image foreground overlap, with empty/empty equal to 1. Test is evaluated after checkpoint selection, not used for threshold tuning. See ../MODEL_CARD.md for limitations and the full-width unverified preset.